# Generazione di un dataset avanzato usando Gemma 4

# Descrizione del problema

All'interno di questo Jupyter Notebook è presente il contenuto del punto 6 del [Notebook 1: Generazione Dataset Elementare](01_Generazione_Dataset_Elementare.ipynb)

Per la generazione di un dataset avanzato ho deciso di chiedere l'aiuto delle LLM. Invece di optare per una normale query all'interno di interfacce web commerciali (come Gemini o soluzioni simili), ho deciso di utilizzare uno strumento molto più versatile e professionale, ovvero Gemma 4. Questo è un modello linguistico di grandi dimensione (Large Language Model) sviluppato da Google che ho configurato ed eseguito localmente tramite la piattaforma Ollama. Questa architettura ha permesso di aggirare i vincoli legati al rate limiting e i costi di consumo tipici delle API basate su cloud (come ad esempio Gemini, anch'essa di Google), garantendo al contempo il controllo totale sui parametri di generazione (come ad esempio la formattazione nativa della risposta). Per maggiori informazioni, è possibile consultare la documentazione ufficiale di [Gemma 4](https://ai.google.dev/gemma/docs/core/model_card_4?hl=it).

In questo dataset ho deciso appositamente di ignorare la creazione di finti nomi e finte email in quanto non richieste dal PW.

### Come eseguire il codice di questo Jupyter Notebook

__Questo Jupyter Notebook contiene già del codice eseguito a scopo dimostrativo, tuttavia a seguire è disponibile una breve guida passo-passo qualora si desiderasse configurare l'ambiente ed eseguirlo nuovamente__

Per eseguire correttamente questo Notebook e avviare la generazione del dataset tramite il modello locale, segui questi passaggi preliminari:

- Ollama deve essere installato direttamente sul tuo sistema operativo per poter ospitare il modello in locale.
* **Windows / Mac / Linux:** Scarica e installa l'applicazione ufficiale dal sito [ollama.com](https://ollama.com).
- Dopo di che, va scaricato ed installato il modello. Da powershell, basta runnare il comando: *ollama run gemma4* (9.6 GB richiesti)
- Una volta installato, per rimuovere il modello basta utilizzare il comando: *ollama rm gemma4:e2b*

In [1]:
%pip install pandas ollama pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Sviluppo del codice

Import delle librerie:

In [2]:
import json
import pandas as pd
import random
from ollama import Client
from pydantic import BaseModel, Field

Struttura base dei parametri di generazione:

In [3]:
class TicketSintetico(BaseModel):
    title: str = Field(
        description="Un titolo breve, specifico e realistico."
    )
    body: str = Field(
        description=(
            "Il corpo del ticket. Deve contenere una descrizione breve (1-2 frasi) del problema reale. "
            "NON usare placeholder, parentesi quadre, simboli di markup o istruzioni di compilazione (es. '[Descrivere qui...]'). "
            "Non è obbligatorio inserire nominativi di applicazioni/servizi ma, se lo fai, utilizza nomi verosimili (non usare ad esempio modulo X o endpoint x)"
            "Il testo deve essere testo pronto, discorsivo e già compilato, scritto dal punto di vista dell'utente o del dipendente che sta riscontrando il problema in prima persona"
        )
    )

Funzione della generazione ticket con gemma. Qui viene formattato il prompt della richiesta da fare a Gemma per la creazione di un determinato ticket. I parametri sono:
- categoria: la categoria del ticket
- priorità: la priorità del problema
- esperienza: il livello di esperienza nel settore specifico della categoria da parte dell'utente
- colloqualità: il tono da utilizzare per la stesura del ticket
- nome_utente: informazione extra da utilizzare in maniera facoltativa che rappresenta se voler aggiungere il nome dell'utente durante la stesura del ticket
- typo: booleano che specifica se richiedere l'inserimento di un typo o meno

Il prompt finale nasce da diverse prove al fine di ottenere un risultato più realistico possibile

L'output della funzione sarà una tupla, contenente body e il title del ticket.

In [4]:
client = Client()
def genera_ticket_con_gemma(categoria, priorita,esperienza,colloqualità,incipit,titolo,typo=False):
    prompt_utente = f"""Genera un ticket di assistenza aziendale in lingua italiana con queste caratteristiche:
    - Categoria del ticket: {categoria}.
    - Priorità del problema: {priorita}
    - Livello di esperienza dell'utente: {esperienza}. Questo rappresenta quanto bene conosce l'argomento della categoria di cui sta parlando.
    - Livello di colloquialità/tono: {colloqualità}

    STILE DI SCRITTURA:
    1. Sii estremamente sintetico. Il testo deve essere composto da massimo 2-3 frasi corte.
    2. Scrivi in prima persona. 
    3. Non usare MAI caratteri speciali di formattazione come asterischi (*), corsivi o grassetti.
    4. Evita deliri o flussi di coscienza lunghi: vai dritto al problema reale.
    5. Se lo ritieni naturale per il tono richiesto, puoi includere il nome dell'utente (da te inventato) nel testo.
    6. Forma di saluto iniziale deve essere TASSATIVAMENTE {incipit} o una sua modifica lieve
    7. Se lo ritieni opportuno in base al tono o al livello di esperienza, fai in modo che il titolo del ticket sia più o meno complesso, non limitarti ad usare parole standard
    8. Il titolo deve simulare l'oggetto scritto da un utente reale (breve, a volte sbrigativo o imperfetto). Inizia il titolo UTILIZZANDO OBBLIGATORIAMENTE questa esatta espressione iniziale: "{titolo}". 
      Continua poi la frase completando l'oggetto in modo che si colleghi perfettamente (es. se il titolo è "Non va ", il titolo diventerà "Non va la posta elettronica" o "Non va internet").
    """
    if typo:
        prompt_utente += "\n- Inserisci dove rendi opportuno un typo per rendere il ticket più human like."

    try:
        response = client.chat(
            model='gemma4', 
            messages=[{'role': 'user', 'content': prompt_utente}],
            format={
                'type': 'object',
                'properties': {
                    'title': {'type': 'string'},
                    'body': {'type': 'string'}
                },
                'required': ['title', 'body']
            },
            options={
                'keep_alive': -1,  # -1 dice a Ollama: "Non scaricare MAI il modello finché non si chiude lo script"
                'temperature': 0.4
            }
        )
        dati_ticket = json.loads(response['message']['content'].strip())
        return dati_ticket['title'], dati_ticket['body']
    except Exception as e:
        print(f"Errore durante la generazione locale: {e}")
        return None, None


Semplicissimo ciclo for responsabile della creazione di *da_fare* ticket.

Componenti fondamentali:
- Definizione dei Pool di Input:
    - Delle semplicissime liste contenenti le categorie e le priorità richieste dalla consegna del PW, in aggiunta dei vari livelli di esperienza e colloqualità, per garantire una maggiore differenziazione dei contenuti
- L'interno del ciclo for
    - All'inizio è possibile notare la prensenza di vari random.choices pesati per creare una logica realistica dei dati che verranno realizzati
    - *titolo, corpo = genera_ticket_con_gemma(categoria, priorità,esperienza,colloqualità,nome,attiva_typo)*, chiama la funzione precedentemente descritta e conserva all'interno delle variabili *titolo* e *corpo* il risultato
    - Viene creato un dizionario e convertito in un dataframe che verrà inserito in all'interno del file csv (in append o in write, in base al fatto se esista già o meno un record al suo interno)


In [8]:
import pandas as pd
import random

percorso_csv = "output/ticket_utenti_gemma.csv"
categorie_disponibili = ["Tecnico", "Amministrazione", "Commerciale"]
priorita_disponibili = ["Bassa", "Media", "Alta"]
livelli_esperienza = ["Principiante", "Intermedio", "Esperto/Professionale"]
livelli_colloquialita = ["Formale e distaccato", "Standard lavorativo", "Molto informale e colloquiale", "Frustrato/Arrabbiato"]
pool_incipit = ["","Ciao, ","Eccomi di nuovo, ","Salve, ","Scusate il disturbo, ","Ho un problema: ","Vi scrivo perché ","Non so più che fare... ","Così non posso lavoare! "]
pool_incipit_titolo = [
    "",                                # 30% Libero (Gemma decide da sola)
    "Non funziona ",                   # Azione diretta
    "Problema con ",                   # Classico ma colloquiale
    "Aiuto: ",                         # Panico / Urgenza
    "Errore ",                         # Tecnico
    "Bloccato su ",                    # Frustrato
    "Info per ",                       # Amministrativo/Commerciale
    "Richiesta urgente: ",             # Alta priorità
    "Non va ",                         # Molto colloquiale/Sbrigativo
    "Chiarimento su "                  # Formale/Fatture
]
da_fare = 5

#controllo se è il primo inserimento o meno
try:
    df_esistente = pd.read_csv(percorso_csv)
    
    if not df_esistente.empty and 'id' in df_esistente.columns:
        ultimo_id_str = df_esistente['id'].iloc[-1]
        id_di_partenza = int(ultimo_id_str.split("-")[1]) + 1
    else:
        id_di_partenza = 0
        
    scrivi_header = False
    print(f"File CSV rilevato con Pandas. Riprendo dall'ID: TK-{id_di_partenza}")

except (FileNotFoundError, pd.errors.EmptyDataError):
    #parto da zero se il file non è stato trovato o è vuoto
    id_di_partenza = 0
    scrivi_header = True
    print("File CSV non trovato o vuoto. Creazione nuovo dataset da TK-0.")

print("-" * 50)

for i in range(0, da_fare):
    id_corrente_numerico = id_di_partenza + i
    stringa_id = f"TK-{id_corrente_numerico}"
    
    categoria = random.choice(categorie_disponibili)
    priorità = random.choices(priorita_disponibili, weights=[50, 30, 20])[0]
    esperienza = random.choices(livelli_esperienza, weights=[40, 40, 20])[0]
    colloqualità = random.choices(livelli_colloquialita, weights=[40, 35, 20, 5])[0]
    attiva_typo = random.choices([True, False], weights=[20, 80])[0]
    incipit = random.choice(pool_incipit)
    titolo = random.choice(pool_incipit_titolo)

    print(f"Generazione in corso... [{categoria} - {priorità} - {esperienza} - {colloqualità} - {attiva_typo}] ({i+1}/{da_fare})")
            
    titolo, corpo = genera_ticket_con_gemma(categoria, priorità, esperienza, colloqualità,incipit,titolo, attiva_typo)
    
    if titolo and corpo:
        singolo_ticket = {
            "id": [stringa_id],
            "title": [titolo],
            "body": [corpo],
            "category": [categoria],
            "priority": [priorità]
        }
        df_singolo = pd.DataFrame(singolo_ticket)
        
        #se è il primo ticket assoluto creato da zero, scrive l'header e apre in 'w'
        if id_corrente_numerico == 0 and scrivi_header:
            df_singolo.to_csv(percorso_csv, index=False, mode='w', header=True)
            scrivi_header = False 
        else:
            df_singolo.to_csv(percorso_csv, index=False, mode='a', header=False)
            
        print(f"Ticket {stringa_id} Creato e SALVATO nel CSV!")
        print(f"Titolo: {titolo}")
        print(f"Corpo:  {corpo[:80]}...") 
        print("-" * 50)
        print()
    else:
        print("Generazione fallita. Salto al successivo.")
        print("-" * 50)

print(f"\nGenerazione conclusa con successo!")
print(f"Trovi tutti i dati aggiornati in tempo reale qui: '{percorso_csv}'.")

File CSV rilevato con Pandas. Riprendo dall'ID: TK-33
--------------------------------------------------
Generazione in corso... [Amministrazione - Media - Principiante - Molto informale e colloquiale - False] (1/5)
Ticket TK-33 Creato e SALVATO nel CSV!
Titolo: Chiarimento su come funziona lo scontrino digitale
Corpo:  Vi scrivo perché non capisco bene le cose relative agli scontrini digitali. Pote...
--------------------------------------------------

Generazione in corso... [Amministrazione - Bassa - Esperto/Professionale - Frustrato/Arrabbiato - False] (2/5)
Ticket TK-34 Creato e SALVATO nel CSV!
Titolo: Info per lo stato del rimborso fiscale
Corpo:  Ho un problema: Mi sembra che la procedura di reso sia bloccata da giorni. Non c...
--------------------------------------------------

Generazione in corso... [Tecnico - Media - Principiante - Standard lavorativo - False] (3/5)
Ticket TK-35 Creato e SALVATO nel CSV!
Titolo: Problema con accesso al sistema CRM dopo aggiornamento
Corpo:

# Conclusioni

- Inizialmente la scelta era ricaduta sulla versione standard e completa del modello Gemma 4. Sebbene l'accuratezza e la diversificazione dei testi fossero ottimali, ho notato una complessità computazionale non da ignorare ed sono andato alla ricerca di alternative più leggere e sostenibili localmente.

- Ho pertanto sperimentato l'introduzione di varianti ottimizzate e quantizzate a parametri ridotti, nello specifico le release da 2 miliardi (e2b) e 4 miliardi di parametri (e4b). Sebbene queste architetture abbiano garantito un'esecuzione radicalmente più fulminea e un'impronta di memoria minima sul sistema, i test sul campo hanno evidenziato una forte "pigrizia semantica"; si sono presentati fenomeni di ripetitività strutturale, generando schemi sintattici quasi identici per i ticket appartenenti alle medesime categorie (quasi nulla di diverso dai ticket generati nel Jupyter 01).

- Poiché l'eterogeneità e la variabilità semantica del dataset sono requisiti fondamentali per evitare l'overfitting e garantire una corretta generalizzazione dei successivi classificatori di Machine Learning, ho stabilito che la qualità del dato dovesse prevalere sulle tempistiche di computazione. Sono dunque ritornato definitivamente al modello base standard di Gemma 4, accettando requisiti hardware più stringenti in cambio di un dataset testuale autentico, eterogeneo e strutturalmente solido.

Diamo un occhiata ai ticket generati da Gemma per validarne la qualità prima di procedere con la generazione del numero di ticket richiesto dal PW.

In [6]:
df_anteprima = pd.read_csv("output/ticket_utenti_gemma.csv")
print("*** TICKET GEMMA ***")
df_anteprima.head(5)["body"]

*** TICKET GEMMA ***


0    Ciao, vorrei richiedere un chiarimento relativ...
1    Vi scrivo perché ho un dubbio sui prezzi che m...
2    Vi scrivo perché non riesco ad accedere alla s...
3    Buongiorno, ho notato che alcuni articoli nel ...
4    Salve, scusa ma non riesco ad accedere al port...
Name: body, dtype: str

__Una EDA più accurata è disponibile all'interno del Jupyter 03.__